### 1.1.3. Set Membership Operations

Set membership operations answer questions about elements and sets:
- **Is element *x* a member of set *S*?** — tested with `in` / `not in`
- **Is set *A* a subset of set *B*?** — tested with `within` / `not within`
- **How many members does *S* have?** — computed with `card(S)`
- **What set results from evaluating expression *e* over an index set?** — built with `setof`

These operations appear in constraint conditions, filtered indexing expressions, `check` statements, `display` commands, and derived set definitions.

#### Formal Syntax

In [ ]:
# Membership operator (element in set)
member  in sexpr        # true if member is in the set
member  not in sexpr    # true if member is not in the set

# Subset operator (set within set)
sexpr within  sexpr     # true if left set is a subset of right set
sexpr not within  sexpr # true if left set is NOT a subset

# Cardinality function
card( sexpr )           # integer: number of members in sexpr

# Set-of operator
setof { indexing } member   # set of all values of member over indexing

**Operand requirements:**
- `in` / `not in`: the left operand is a **scalar expression** (a number, string, or parenthesised tuple) whose dimension equals the dimension of the right operand set. The right operand is a **set expression**.
- `within` / `not within`: both operands are **set expressions** of the same dimension.
- `card`: the argument is any **set expression** (including an inline indexing expression).
- `setof`: the `indexing` is an AMPL indexing expression; or a comma-separated list of expressions enclosed in parentheses. The resulting set consists of all members obtained by iterating over the indexing expression;

#### The `in` and `not in` Operators

The `in` operator is the AMPL counterpart of the mathematical set-membership symbol $\in$. It tests whether a **scalar value** (or tuple) belongs to a set.

- `expr in S` evaluates to `1` (true) if the value of `expr` is a member of `S`, and `0` (false) otherwise.
- `expr not in S` is the logical negation: true when the value is absent from `S`.

The left operand must be a scalar whose type and dimension match those of the set. For a one-dimensional set of strings, the left operand must be a string; for a set of pairs, it must be a parenthesised pair of expressions.

Typical uses:
- **Filter condition in an indexing expression**: `{i in NUTR: i in MAXREQ}` restricts iteration to nutrients that also appear in `MAXREQ`.
- **Conditional constraint via `if`**: selectively apply a constraint body depending on set membership.

In [ ]:
set NUTR;                        # all nutrients
set MAXREQ within NUTR;          # nutrients with an upper bound
set MINREQ within NUTR;          # nutrients with a lower bound
set FOOD;
param amt {NUTR, FOOD} >= 0;
param n_min {MINREQ} >= 0;
param n_max {MAXREQ} >= 0;
var Buy {FOOD} >= 0;

# Use 1: filter condition — iterate only over nutrients that have a maximum bound
set NUTREQ = {i in NUTR: i in MAXREQ or i in MINREQ};

# Use 2: if condition in constraint — apply upper bound only when applicable
subject to DietMax {i in NUTR: i in MAXREQ}:
    sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

#### The `within` and `not within` Operators

The `within` operator is the AMPL counterpart of the subset symbol $\subseteq$. It tests whether every member of one **set** is also a member of another.

- `A within B` is true if and only if every element of `A` is also in `B` (i.e., $A \subseteq B$).
- `A not within B` is the negation.

Both operands must be set expressions of the same dimension.

In [ ]:
set NUTR;                       # all nutrients
set MINREQ within NUTR;         # attribute form: enforces MINREQ subset of NUTR
set MAXREQ within NUTR;         # attribute form: enforces MAXREQ subset of NUTR

# Operator form in an if-then-else expression
param overlap_count = if MINREQ within MAXREQ then card(MINREQ) else 0;

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
The left operand of <code>in</code> must be a <em>scalar expression</em> (a number, string, or parenthesised tuple), and the left operand of <code>within</code> must be a <em>set</em>. Swapping them is a type error. For example, <code>MINREQ in NUTR</code> is illegal because <code>MINREQ</code> is a set, not a scalar — the correct form is <code>MINREQ within NUTR</code>. Conversely, <code>"B2" within NUTR</code> is illegal because <code>"B2"</code> is a string scalar, not a set — the correct form is <code>"B2" in NUTR</code>.</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use the <code>within</code> <em>attribute</em> in set declarations to enforce subset constraints at data-load time rather than relying on <code>check</code> statements or runtime <code>within</code> operator tests. A declaration such as <code>set MAXREQ within NUTR;</code> causes AMPL to validate each data element immediately when the data section is read, producing an informative error before the model is ever solved.</div>

#### The `card()` Function

The built-in function `card` computes the **cardinality** (number of members) of a set.

- `card(S)` returns an integer equal to the number of members currently in `S`.
- The argument may be any expression that evaluates to a set, including an inline indexing expression.
- When the argument is an indexing expression (not just a set name), the parentheses around `card`'s argument may be omitted per AMPL's grammar, but including them is clearer.

`card` is commonly used to:
- Define a parameter whose value depends on the size of a set.
- Bound a parameter or variable by the number of available resources.
- State a constraint that limits how many set members are active.

In [ ]:
set NUTR;
set FOOD;
set MAXREQ within NUTR;
param amt {NUTR, FOOD} >= 0;
var Buy {FOOD} >= 0;

# Use 1: scalar param derived from set size
param n_nutr = card(NUTR);            # number of nutrients

# Use 2: upper bound on a parameter using set cardinality
param budget >= 0, <= card(FOOD);     # budget cannot exceed number of foods

# Use 3: cardinality of a filtered (inline) set expression
param n_bounded = card({i in NUTR: i in MAXREQ});  # count nutrients with upper bound

#### The `setof` Operator

The `setof` operator constructs a new set by evaluating a **member expression** for each element of an indexing set:

```python
setof { indexing } member
```

- `indexing` is an AMPL indexing expression; or a comma-separated list of expressions enclosed in parentheses.  
- `member` is a scalar expression or a parenthesised tuple of expressions.
- The result is the set of all distinct values produced by evaluating `member` over `indexing`. Duplicate values are silently dropped.
- The dimension of the result equals the number of components in `member`.

`setof` is the AMPL analogue of a set comprehension that maps index tuples to computed values. It is distinct from a filtered indexing expression `{i in S: condition}`, which collects the *index values themselves* rather than the value of a separate expression.

In [ ]:
set PROD;                          # products
param cost {PROD} >= 0;            # cost per unit
var Make {PROD} >= 0;
var Sell {PROD} >= 0;

# Collect the distinct cost values that appear across all products
set COST_VALUES = setof {p in PROD} cost[p];

# Collect the set of products that have a positive sales variable bound
# (parameter context: using an upper bound param, not a variable)
param market {PROD} >= 0;
set ACTIVE_PROD = setof {p in PROD: market[p] > 0} p;

In [ ]:
set ROUTES dimen 3;               # set of (orig, dest, prod) triples

# Derive the set of products that appear in at least one route
set PROD = setof {(i,j,p) in ROUTES} p;

# Derive the set of used origin-destination pairs
set LINKS = setof {(i,j,p) in ROUTES} (i,j);

#### Edge Cases

**Edge case 1: `card` of an empty set is zero, not an error.**

If the set expression evaluates to the empty set, `card` returns `0`. This is well-defined and does not raise an error. However, using `card(S)` as a divisor or as the upper bound of a non-trivial constraint requires guarding against the zero case.

In [ ]:
set PROD;
param cost {PROD} >= 0;

# EXPENSIVE is empty when no product costs more than 100
set EXPENSIVE = {p in PROD: cost[p] > 100};

# card(EXPENSIVE) = 0 when EXPENSIVE is empty -- valid, not an error
param n_expensive = card(EXPENSIVE);

# Guard against division by zero before using cardinality as a divisor
param avg_expensive_cost = if card(EXPENSIVE) > 0
    then (sum {p in EXPENSIVE} cost[p]) / card(EXPENSIVE) else 0;

**Edge case 2: `setof` silently drops duplicates.**

When two or more members of the indexing set produce the same value for the `member` expression, the result set contains that value only once. This is by definition (sets have distinct elements), but it means `card(setof {i in S} f[i])` may be strictly less than `card(S)` when `f` is not injective.

In [ ]:
set PROD;
param cost {PROD} >= 0;

# If two products share the same cost, COST_VALUES has fewer members than PROD
set COST_VALUES = setof {p in PROD} cost[p];